# 02 Preprocessing - Data Quality Checks

This notebook performs an initial data-quality audit on the raw dataset before preprocessing.

Checks included:
- Missing values
- Data type validation against expected schema
- Suspicious/out-of-place value checks based on business rules


In [2]:
%pip install pandas

import pathlib
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/usr/local/bin/python3.9 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
data_dir = Path('../data/raw')
csv_files = sorted(data_dir.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError('No CSV files found in ../data/raw')

dataset_path = csv_files[0]
df = pd.read_csv(dataset_path)

print(f'Loaded dataset: {dataset_path.name}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

Loaded dataset: ElectraHub_Data.csv
Shape: 3,000 rows x 16 columns


## 1) Missing Values

In [4]:
missing_by_col = df.isna().sum().rename('missing_count').to_frame()
missing_by_col['missing_pct'] = (missing_by_col['missing_count'] / len(df) * 100).round(2)

total_missing = int(missing_by_col['missing_count'].sum())
print(f'Total missing values in dataset: {total_missing}')

missing_by_col

Total missing values in dataset: 0


,missing_count,missing_pct
Region,0,0.0
Product_Category,0,0.0
Campaign_Type,0,0.0
Product_Age_Months,0,0.0
Product_Price,0,0.0
Competitor_Price_Index,0,0.0
Advertising_Expenditure,0,0.0
Discount_Percentage,0,0.0
Campaign_Engagement_Score,0,0.0
Inventory_Level,0,0.0


## 2) Data Type Validation

In [5]:
expected_types = {
    'Region': 'object',
    'Product_Category': 'object',
    'Campaign_Type': 'object',
    'Product_Age_Months': 'int64',
    'Product_Price': 'float64',
    'Competitor_Price_Index': 'float64',
    'Advertising_Expenditure': 'float64',
    'Discount_Percentage': 'float64',
    'Campaign_Engagement_Score': 'float64',
    'Inventory_Level': 'int64',
    'Num_Reviews': 'int64',
    'Avg_Customer_Rating': 'float64',
    'Return_Rate': 'float64',
    'Length_Product_Description': 'int64',
    'Popularity': 'object',
    'Sales': 'float64'
}

actual_types = {col: str(dtype) for col, dtype in df.dtypes.items()}

dtype_check = pd.DataFrame({
    'column': df.columns,
    'expected_dtype': [expected_types.get(c, 'N/A') for c in df.columns],
    'actual_dtype': [actual_types.get(c, 'N/A') for c in df.columns]
})
dtype_check['status'] = np.where(dtype_check['expected_dtype'] == dtype_check['actual_dtype'], 'OK', 'CHECK')

dtype_check

,column,expected_dtype,actual_dtype,status
0,Region,object,object,OK
1,Product_Category,object,object,OK
2,Campaign_Type,object,object,OK
3,Product_Age_Months,int64,int64,OK
4,Product_Price,float64,float64,OK
5,Competitor_Price_Index,float64,float64,OK
6,Advertising_Expenditure,float64,float64,OK
7,Discount_Percentage,float64,float64,OK
8,Campaign_Engagement_Score,float64,float64,OK
9,Inventory_Level,int64,int64,OK


## 3) Suspicious or Out-of-Place Values

In [6]:
issues = []

# Categorical expectations
expected_regions = {'North', 'South', 'East', 'West'}
expected_categories = {'Mobile', 'Tablet'}
expected_campaign_types = {'Email', 'Search Ad', 'Social Media', 'Influencer'}
expected_popularity = {'Low', 'Moderate', 'High', 'Very High', 'Very Low'}

cat_rules = [
    ('Region', expected_regions),
    ('Product_Category', expected_categories),
    ('Campaign_Type', expected_campaign_types),
    ('Popularity', expected_popularity)
]

for col, allowed in cat_rules:
    unexpected = sorted(set(df[col].dropna().unique()) - allowed)
    issues.append({
        'column': col,
        'check': 'Unexpected category values',
        'issue_count': len(unexpected),
        'details': ', '.join(unexpected) if unexpected else 'None'
    })

# Numeric business-rule checks
numeric_checks = [
    ('Product_Age_Months', (df['Product_Age_Months'] < 0).sum(), 'Age in months should be >= 0'),
    ('Product_Price', (df['Product_Price'] <= 0).sum(), 'Price should be > 0'),
    ('Competitor_Price_Index', (df['Competitor_Price_Index'] <= 0).sum(), 'Price index should be > 0'),
    ('Advertising_Expenditure', (df['Advertising_Expenditure'] < 0).sum(), 'Ad spend should be >= 0'),
    ('Discount_Percentage', ((df['Discount_Percentage'] < 0) | (df['Discount_Percentage'] > 100)).sum(), 'Discount should be between 0 and 100'),
    ('Campaign_Engagement_Score', ((df['Campaign_Engagement_Score'] < 0) | (df['Campaign_Engagement_Score'] > 100)).sum(), 'Engagement score should be between 0 and 100'),
    ('Inventory_Level', (df['Inventory_Level'] < 0).sum(), 'Inventory should be >= 0'),
    ('Num_Reviews', (df['Num_Reviews'] < 0).sum(), 'Review count should be >= 0'),
    ('Avg_Customer_Rating', ((df['Avg_Customer_Rating'] < 1) | (df['Avg_Customer_Rating'] > 5)).sum(), 'Rating expected in 1-5 range'),
    ('Return_Rate', ((df['Return_Rate'] < 0) | (df['Return_Rate'] > 100)).sum(), 'Return rate should be between 0 and 100'),
    ('Length_Product_Description', (df['Length_Product_Description'] <= 0).sum(), 'Description length should be > 0'),
    ('Sales', (df['Sales'] < 0).sum(), 'Sales should be >= 0')
]

for col, cnt, desc in numeric_checks:
    issues.append({
        'column': col,
        'check': desc,
        'issue_count': int(cnt),
        'details': 'None' if int(cnt) == 0 else 'Has rule violations'
    })

# Outlier signal using IQR (not necessarily errors, but worth reviewing)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = int(((df[col] < lower) | (df[col] > upper)).sum())
    issues.append({
        'column': col,
        'check': 'IQR outlier count (review only)',
        'issue_count': outliers,
        'details': f'Lower<{lower:.3f} or Upper>{upper:.3f}'
    })

issues_df = pd.DataFrame(issues)
issues_df.sort_values(['issue_count', 'column'], ascending=[False, True]).reset_index(drop=True)

,column,check,issue_count,details
0,Num_Reviews,IQR outlier count (review only),119,Lower<-301.000 or Upper>1059.000
1,Product_Age_Months,IQR outlier count (review only),113,Lower<-10.000 or Upper>30.000
2,Return_Rate,IQR outlier count (review only),22,Lower<1.624 or Upper>3.897
3,Length_Product_Description,IQR outlier count (review only),21,Lower<80.000 or Upper>416.000
4,Avg_Customer_Rating,IQR outlier count (review only),17,Lower<3.620 or Upper>4.340
5,Sales,IQR outlier count (review only),16,Lower<7243.054 or Upper>45587.644
6,Campaign_Engagement_Score,IQR outlier count (review only),15,Lower<11.795 or Upper>93.035
7,Inventory_Level,IQR outlier count (review only),12,Lower<37.125 or Upper>1610.125
8,Competitor_Price_Index,IQR outlier count (review only),9,Lower<0.777 or Upper>1.221
9,Discount_Percentage,IQR outlier count (review only),7,Lower<14.970 or Upper>43.930


## 4) Summary Findings

In [7]:
summary_notes = []

total_missing = int(df.isna().sum().sum())
summary_notes.append(f'- Missing values: **{total_missing}** total across all columns.')

dtype_mismatches = int((dtype_check['status'] == 'CHECK').sum())
summary_notes.append(f'- Data type mismatches vs expected schema: **{dtype_mismatches}**.')

rule_violations = issues_df[
    (~issues_df['check'].str.contains('IQR outlier')) &
    (issues_df['issue_count'] > 0)
]
summary_notes.append(f'- Business-rule violations detected: **{len(rule_violations)} columns with violations**.')

top_outlier_cols = issues_df[
    issues_df['check'].str.contains('IQR outlier')
].sort_values('issue_count', ascending=False).head(5)

if top_outlier_cols['issue_count'].max() > 0:
    outlier_text = ', '.join([f"{r.column} ({int(r.issue_count)})" for r in top_outlier_cols.itertuples(index=False)])
    summary_notes.append(f'- Most prominent outlier signals (IQR): {outlier_text}.')
else:
    summary_notes.append('- No IQR outlier signals found in numeric columns.')

print('\n'.join(summary_notes))

- Missing values: **0** total across all columns.
- Data type mismatches vs expected schema: **0**.
- Business-rule violations detected: **0 columns with violations**.
- Most prominent outlier signals (IQR): Num_Reviews (119), Product_Age_Months (113), Return_Rate (22), Length_Product_Description (21), Avg_Customer_Rating (17).


In [8]:
# Helpful quick-reference profile
display(df.describe(include='all').transpose())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,3000,4,West,845,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Category,3000,2,Mobile,1884,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Campaign_Type,3000,4,Social Media,937,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Age_Months,3000.0,NaN,NaN,NaN,11.316,8.289695,1.0,5.0,9.0,15.0,36.0
Product_Price,3000.0,NaN,NaN,NaN,980.839717,571.79498,10.11,526.575,906.935,1475.72,2000.0
Competitor_Price_Index,3000.0,NaN,NaN,NaN,0.999106,0.080977,0.78,0.944,0.998,1.055,1.25
Advertising_Expenditure,3000.0,NaN,NaN,NaN,623.781673,84.686263,325.34,564.3575,623.8,683.4,820.0
Discount_Percentage,3000.0,NaN,NaN,NaN,29.43954,5.260201,9.25,25.83,29.535,33.07,42.0
Campaign_Engagement_Score,3000.0,NaN,NaN,NaN,52.344707,14.614951,5.72,42.26,52.19,62.57,99.0
Inventory_Level,3000.0,NaN,NaN,NaN,821.361333,290.326387,50.0,627.0,822.0,1020.25,1840.0


## 5) Proposed Cleaning Plan (Review Before Execution)

This plan is based on the quality audit findings above. It is intentionally conservative because no hard data-integrity errors were found.

| Issue Category | What We Found | Recommended Action | Why This Recommendation |
|---|---|---|---|
| Missing values | No missing values in any column. | Keep current values as-is; do not apply imputation. Add a missing-value check in the pipeline to catch future drift. | Avoids unnecessary transformations and keeps preprocessing minimal/reproducible. |
| Data types | All columns match expected types. | Keep current dtypes, but enforce schema at load time (assert expected columns and dtypes). | Prevents silent schema drift when new data arrives. |
| Unexpected categorical labels | No unexpected values in `Region`, `Product_Category`, `Campaign_Type`, `Popularity`. | Keep categories as-is. During encoding, set `handle_unknown='ignore'` for robustness. | Training data is clean today, but production/inference data may include unseen categories. |
| Business-rule violations | No invalid values for price/rates/inventory/reviews/sales bounds. | Keep values; add lightweight rule assertions before model training. | Early failure is better than training with invalid future records. |
| Outlier signals (IQR) | Outlier counts appear in `Num_Reviews`, `Product_Age_Months`, `Return_Rate`, `Length_Product_Description`, `Avg_Customer_Rating`, `Sales`. | Do not drop rows immediately. Start with robust scaling/winsorization candidates only for heavily skewed predictors (for example `Num_Reviews`). Compare model performance with and without clipping. | These may be real business extremes, not errors. Removing them blindly can hurt model realism. |
| Target variable (`Sales`) extremes | A small number of high/low `Sales` outliers. | Keep target values. Optionally test `log1p(Sales)` variant for sensitivity analysis in modeling stage. | Linear regression can be sensitive to target skew; comparing transformed vs raw target is safer than hard deletion. |
| Duplicate rows | No exact duplicates. | No deduplication needed right now; keep duplicate check in pipeline. | Maintains data integrity checks for future batches. |

### Recommended Execution Order (When You Approve)

1. Freeze schema checks (column set + dtypes + basic rule assertions).
2. Split features/target and train/validation sets.
3. Build preprocessing with:
   - Numeric: optional scaler (`StandardScaler` or `RobustScaler`).
   - Categorical: `OneHotEncoder(handle_unknown='ignore')`.
4. Run two variants for comparison:
   - Baseline: no outlier clipping.
   - Sensitivity: winsorize/clip selected numeric predictors.
5. Compare validation metrics and keep the simplest stable approach.

### What Not To Do Yet

- Do not remove outliers globally.
- Do not impute values when missingness is zero.
- Do not engineer complex transformations before baseline regression is established.


## 6) Apply Cleaning Steps

This section applies only the approved cleaning actions:
- Enforce schema and basic integrity checks
- Standardize text fields (trim outer whitespace)
- Drop exact duplicate rows if present
- Re-check business-rule validity
- Save cleaned dataset to `../data/processed/`

No encoding, scaling, or feature engineering is applied here.


In [9]:
# Start from raw data copy
clean_df = df.copy()

# 1) Standardize text fields: trim leading/trailing whitespace
text_cols = clean_df.select_dtypes(include=['object']).columns.tolist()
for col in text_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# 2) Enforce expected schema (columns + dtypes)
expected_columns = list(expected_types.keys())
missing_cols = [c for c in expected_columns if c not in clean_df.columns]
extra_cols = [c for c in clean_df.columns if c not in expected_columns]

if missing_cols:
    raise ValueError(f'Missing expected columns: {missing_cols}')
if extra_cols:
    raise ValueError(f'Unexpected extra columns: {extra_cols}')

clean_df = clean_df[expected_columns]

for col, dtype in expected_types.items():
    clean_df[col] = clean_df[col].astype(dtype)

# 3) Drop exact duplicate rows (if any)
before_rows = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
after_rows = len(clean_df)

# 4) Re-check key business rules
rule_failures = {
    'Product_Age_Months_negative': int((clean_df['Product_Age_Months'] < 0).sum()),
    'Product_Price_non_positive': int((clean_df['Product_Price'] <= 0).sum()),
    'Competitor_Price_Index_non_positive': int((clean_df['Competitor_Price_Index'] <= 0).sum()),
    'Advertising_Expenditure_negative': int((clean_df['Advertising_Expenditure'] < 0).sum()),
    'Discount_Percentage_out_of_0_100': int(((clean_df['Discount_Percentage'] < 0) | (clean_df['Discount_Percentage'] > 100)).sum()),
    'Campaign_Engagement_Score_out_of_0_100': int(((clean_df['Campaign_Engagement_Score'] < 0) | (clean_df['Campaign_Engagement_Score'] > 100)).sum()),
    'Inventory_Level_negative': int((clean_df['Inventory_Level'] < 0).sum()),
    'Num_Reviews_negative': int((clean_df['Num_Reviews'] < 0).sum()),
    'Avg_Customer_Rating_out_of_1_5': int(((clean_df['Avg_Customer_Rating'] < 1) | (clean_df['Avg_Customer_Rating'] > 5)).sum()),
    'Return_Rate_out_of_0_100': int(((clean_df['Return_Rate'] < 0) | (clean_df['Return_Rate'] > 100)).sum()),
    'Length_Product_Description_non_positive': int((clean_df['Length_Product_Description'] <= 0).sum()),
    'Sales_negative': int((clean_df['Sales'] < 0).sum())
}

failed_rules = {k: v for k, v in rule_failures.items() if v > 0}
if failed_rules:
    raise ValueError(f'Business-rule violations remain after cleaning: {failed_rules}')

# 5) Save cleaned data
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
out_path = processed_dir / 'ElectraHub_Data_cleaned.csv'
clean_df.to_csv(out_path, index=False)

print(f'Removed duplicate rows: {before_rows - after_rows}')
print(f'Cleaned dataset saved to: {out_path}')
print(f'Cleaned shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]:,} columns')
print('Column dtypes after cleaning:')
print(clean_df.dtypes)


Removed duplicate rows: 0
Cleaned dataset saved to: ../data/processed/ElectraHub_Data_cleaned.csv
Cleaned shape: 3,000 rows x 16 columns
Column dtypes after cleaning:
Region                         object
Product_Category               object
Campaign_Type                  object
Product_Age_Months              int64
Product_Price                 float64
Competitor_Price_Index        float64
Advertising_Expenditure       float64
Discount_Percentage           float64
Campaign_Engagement_Score     float64
Inventory_Level                 int64
Num_Reviews                     int64
Avg_Customer_Rating           float64
Return_Rate                   float64
Length_Product_Description      int64
Popularity                     object
Sales                         float64
dtype: object
